In [1]:
#!pip install spectral-cube
#!pip install astroquery
#!pip install reproject
!pip install ccdproc
!pip install astropy pandas
!pip install ipycanvas

In [2]:
import glob #importing glob library
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

import astropy
import astropy.units as u
from astropy.utils.data import download_file
from astropy.io import fits  # We use fits to open the actual data file
from astropy.utils import data
from astropy.wcs import wcs
from astropy.coordinates import SkyCoord
from astropy.time import Time
import pandas as pd
from astropy.nddata import CCDData
from astropy.visualization import (AsinhStretch, LogStretch, 
                                  PercentileInterval, ImageNormalize, simple_norm, ZScaleInterval) #tools for visualizing large range of fits data
from astropy.modeling import models, fitting
from astropy.utils.exceptions import AstropyUserWarning
from astropy.nddata import block_reduce
from ccdproc import combine

import ipywidgets as widgets
from IPython.display import display
import os
import ccdproc
import warnings
import copy

from ipycanvas import Canvas
from PIL import Image
from PIL import ImageTk
from tkinter import *

In [3]:
'Creating paths to files both on a personal computer and external hard drive'
mflat_2048 = 'C:/Users/Astroadmin/mflat_2048.fits'
mflat_512 = 'C:/Users/Astroadmin/mflat_512.fits'

#raw_fits_files = glob.glob('C:/Users/jules/OneDrive/Desktop/Test Flat/*.fit')
#mflat_2048 = 'C:/Users/jules/OneDrive/Desktop/Test Flat/mflat.fits'
#mflat_512 = 'C:/Users/jules/mflat_512.fits'

In [4]:
'Pathing to the folder containing all usable fits files for the project'
all_fits_files = glob.glob('D:/EB NSVS01031772 Usable Files/*.fit')

In [51]:
'Designated path is a variable which may be set to any path which is then called by the rest of the code'
designated_path = 'D:/EB NSVS01031772 Usable Files Trimmed/'

In [52]:
'Defining a function which builds a python dictionary with entries ordered based on date, for the organization of raw fits data'

def file_by_date (directory):
    raw_fits_date = [] 
    fits_times = []

    #os.walk takes a root - a path from the inputted directory to the first subfolder that contains mixing of files and directories.
    #dirs and files store the names of the constituent directories and files respectively - dirs may be appended to a path to find files deeper in the directory.
    for root, dirs, files in os.walk(directory):
        
        #Index through each file and combine its name with the path to its directory to produce a full path legible to python
        for filename in files:
            if filename.endswith(('.fits', '.fit')):
                file_path = os.path.join(root, filename) #In our case, dirs mostly contains useless data so it is not part of file_path
                
                #Pull raw fits data from the headers and metadata of each file
                try:
                    with fits.open(file_path) as hdul:
                        date = hdul[0].header['DATE-OBS']
                        size = hdul[0].header['NAXIS1']

                    t = Time(date, format='isot', scale='utc')
                    MJD_fits = t.mjd #Converting ISO Date format in files to MJD for convenience
            
                    #Store desired metadata in a python dictionary containing data for each file
                    raw_fits_date.append({
                        'File Name': filename,
                        'Date': date,
                        'MJD': float(MJD_fits),
                        'Location': root,
                        'Size': size
                    })                        

                #Protects against cell crashing from non-existent files or directories
                except (KeyError, OSError) as e: 
                    print(f"Skipping {filename}: {e}")

    #lambda x describes a key within the dictionary by which the dictionary is sorted - in this case - 'Date'
    raw_fits_date.sort(key=lambda x: x['Date'])

    return raw_fits_date

raw_fits_dict = file_by_date(designated_path) #Define the dictionary as a variable

In [53]:
def search_file(search_query, raw_fits_dict): #searchable dictionary for the flat for loop
    for i in raw_fits_dict:
        if i.get('File Name') == search_query:
            return os.path.join(i['Location'], i['File Name'])
        
    print(f"Error: Key '{search_query}' not found in the dictionary.")
    return None

search_file('test-0001.fit', raw_fits_dict)

'D:/EB NSVS01031772 Usable Files Trimmed/2022 EB Files\\03-22-2019\\test-0001.fit'

In [54]:
flatfield_select = widgets.Select(
    options=list(dict.fromkeys([i['Date'].split('T')[0] for i in raw_fits_dict])), #Widget which allows selection of a single night over which observations were made
    description='Display:',
    disabled=False
)

In [55]:
def dust (date):
    ccd_flat_list = []
    prev_mjd = None
    
    for i in raw_fits_dict:
        mjd = i['MJD']
        date_matches = i['Date'].split('T')[0] == date+
        date_close = prev_mjd is not None and (0.01180556 <= mjd - prev_mjd <= 0.104167)
        
        if prev_mjd is None and date_matches:
            spec_full_path = os.path.join(i['Location'], i['File Name'])
            ccd_flat_list.append(spec_full_path)
            prev_mjd = mjd
        elif date_close: 
            spec_full_path = os.path.join(i['Location'], i['File Name'])
            ccd_flat_list.append(spec_full_path)
            prev_mjd = mjd

    print(ccd_flat_list)

    if not ccd_flat_list:
        print("No files found matching criteria.")
        return

    if len(ccd_flat_list) > 50:
        step_size = len(ccd_flat_list) // 50
        ccd_flat_list = ccd_flat_list[::step_size]

    median_flats = []
        
    for path in ccd_flat_list:
        with fits.open(path) as hdul:
            median_flats.append(1.0 / np.median(hdul[0].data))

    master_flat = combine(
        ccd_flat_list, 
        method='median', 
        scale=median_flats, 
        mem_limit=2e9, # Keeps memory usage strictly capped at 2 GB
        unit='adu'
    )

    fits_header = fits.Header(master_flat.header)
    clean_hdu = fits.PrimaryHDU(data=master_flat.data, header=fits_header)
    unique_filename = f"mflat_{date}.fits"
    clean_hdu.writeto(unique_filename, overwrite=True)

    nx, ny = master_flat.data.shape
    z = master_flat.data
    x,y = np.mgrid[:nx, :ny]
                 
    # Fitting the data using astropy.modeling
    raw_image = models.Polynomial2D(degree=2)
    fitted_image = fitting.LinearLSQFitter()
    fit = fitted_image(raw_image, x, y, z)

    vmin_val = np.percentile(z, 1)
    vmax_val = np.percentile(z, 99)

    # Plot the data with the best-fit model
    fig, axs = plt.subplots(figsize=(8, 2.5), ncols=3)
    ax1 = axs[0]
    ax1.imshow(z, cmap='gray', origin='lower', interpolation='nearest', vmin=vmin_val, vmax=vmax_val)
    ax1.set_title("Data")
    
    ax2 = axs[1]
    ax2.imshow(fit(x,y), cmap='gray', origin='lower', interpolation='nearest', vmin=vmin_val,
           vmax=vmax_val)
    ax2.set_title("Model")
    
    ax3 = axs[2]
    residual = z - fit(x,y)
    vmin_res = np.percentile(residual, 1)
    vmax_res = np.percentile(residual, 99)
    ax3.imshow(residual, cmap='gray', origin='lower', interpolation='nearest', vmin=vmin_res,
           vmax=vmax_res)
    ax3.set_title("Residual")

select = widgets.interactive(dust, date=flatfield_select) #Ties the widget to the above function
display(select)

interactive(children=(Select(description='Display:', options=('2015-03-25', '2015-03-30', '2015-04-02', '2015-…

In [10]:
date = select.result #Stores the selected date as a variable

In [11]:
warnings.filterwarnings('ignore', category=Warning, append=True)
master_flat_2048 = CCDData.read(mflat_2048, unit='adu', ignore_missing_simple=True)
master_flat_512 = CCDData.read(mflat_512, unit='adu', ignore_missing_simple=True)

#SIZE_512 and SIZE_2048 describes the two possible sizes of our fits files, determined by binning factors - may be added to if necessary
SIZE_512 = 512 
SIZE_2048 = 2048

def flatfield (dictionary, date):
    prev_mjd = None
    processed_data = []

    for i in dictionary:
        mjd = i['MJD']
        date_matches = i['Date'].split('T')[0] == date #If the date matches the selected date, files will be flatfielded
        date_close = prev_mjd is not None and (mjd - prev_mjd < 0.104167) #The date could also technically be from a different night but be from the same observation period

        if date_matches or date_close:
            raw_file_path = search_file(i['File Name'], raw_fits_dict)
            fits_reading = CCDData.read(raw_file_path, unit='adu', ignore_missing_simple=True) #Reading fits files that match the selected date
            corrected_fits = copy.deepcopy(fits_reading)

            if i['Size'] == SIZE_512:
                ccdproc.flat_correct(corrected_fits, flat=master_flat_512) #Flatfielding function for 512x512 fits files
                
            elif i['Size'] == SIZE_2048:
                ccdproc.flat_correct(corrected_fits, flat=master_flat_2048) #Flatfielding function for 2048x2048 fits files

            else:
                print(f'Error: File {i['File Name']} size incompatible') #If no master flat of complementary size exits, pass this string

            #Creating a new dictionary to store pertinent values
            processed_data.append({
                'metadata': i,
                'image data': corrected_fits.data,
                'header': corrected_fits.header
            })     
                
            prev_mjd = mjd

    print(f"Processed {len(processed_data)} images for date: {date}")
    return processed_data

flatfield_return = flatfield(raw_fits_dict, date[0]['Date'])

TypeError: 'NoneType' object is not subscriptable

In [ ]:
def folder_by_date(processed_data):
    current_folder_date = ""
    prev_mjd = None
    folder_paths=[]

    for i in processed_data:
        item = i['metadata']
        mjd = item['MJD']
        flat_hdu = fits.PrimaryHDU(data=i['image data'], header=i['header'])
        
        if prev_mjd is None:
            current_folder_date = Time(mjd, format='mjd').iso.split()[0]
            
        else:
            if mjd - prev_mjd >= 0.104167:
                current_folder_date = Time(mjd, format='mjd').iso.split()[0]
                
        new_dir = f'C:/Users/Astroadmin/Desktop/EB_{current_folder_date}'
        name = f'EB_{current_folder_date}'
        
        current_path_info = {
            'Location': new_dir,
            'Name': name
        }
        
        os.makedirs(new_dir, exist_ok=True)

        if current_path_info not in folder_paths: 
            folder_paths.append(current_path_info)
        
        file_output = os.path.join(new_dir, item['File Name'])
        
        flat_hdu.writeto(file_output, overwrite=True)  
        
        prev_mjd = mjd

    return folder_paths

folder_options = folder_by_date(select.result)

In [ ]:
cube_file_select = widgets.Select(
    options=[i['Name'] for i in folder_options],
    description='Display:',
    disabled=False
)

In [ ]:
def cubed_normed_fits (directory):
    norm_fits = []
    
    for root, dirs, file in os.walk(f'C:/Users/Astroadmin/Desktop/{directory}'):
        for filename in file:
            if filename.endswith(('fit', 'fits')):
                full_path = os.path.join(root, filename)

                try:
                    with fits.open(full_path) as hdu_fits:
                        data_org = hdu_fits[0].data.astype(np.float32)
                        data_max = np.nanmax(data_org)
                        if data_max == 0: 
                            data_max = 1.0
                        data_norm = data_org / data_max
                        norm_fits.append(data_norm)

                except (KeyError, OSError) as e: 
                    print(f"Skipping {filename}: {e}")

    num_frames = len(norm_fits)
    output = select.result[0]['metadata']
    size = output['Size']
    
    cube = np.zeros((num_frames, size, size), dtype=np.float32)

    for i, data_array in enumerate(norm_fits):
                        cube[i,:,:] = data_array

    hdu_norm_new = fits.PrimaryHDU(cube)
    hdu_norm_new.writeto('cube_norm.fits', overwrite=True)

    cube_norm = fits.open('cube_norm.fits', memmap = False) 
    cube_norm_data = cube_norm[0].data

    return cube_norm_data

widgets.interactive(cubed_normed_fits, directory=cube_file_select)

In [ ]:
item = [i['metadata'] for i in date]
mid_file = item[int(len(item)/2)]
display_file_path = f'C:/Users/Astroadmin/Desktop/EB_{item[0]['Date'].split('T')[0]}/{mid_file['File Name']}'

image_data = fits.open(display_file_path)[0].data #reads FITS data from file
interval = ZScaleInterval(n_samples=1000, contrast=0.25) #interval is created by sampling the file
vmin,vmax = interval.get_limits(image_data) #creates min and max value based on defined interval

#This process is necessary because python will just assign the highest value to white and the lowest to black, so most of the details in unedited 
#data will become black due to the insanely high counts for a few very bright stars

if vmax <= vmin or (vmax - vmin) < 1e-5: #safeguard - if the interval is way to big it will just reset to 99%
    interval = PercentileInterval(99)
    vmin, vmax = interval.get_limits(image_data) #the top and bottom 0.5% will be cut off

data_range = vmax - vmin #defining range of data that will be displayed
vmin_adjusted = vmin +(data_range * 0.18) #cutting off more from the bottom depending on how big the range is

clipped_image_data = np.clip(image_data, vmin_adjusted, vmax) #clipping off edges of data based on limits
normalized = (clipped_image_data - vmin_adjusted) / (vmax - vmin_adjusted) #normalization fromula usually done by .plt
canvas_image = (normalized * 255).astype(np.uint8) #this is needed to turn FITS data into color values that canvas can read

In [ ]:
from PIL import Image

root = Tk()
root.title("Pick Star")
root.geometry('900x700') #creating basic features of the canvas

pil_img = Image.fromarray(canvas_image, mode='L') #turns array into canvas image, L means greyscale for some reason
tk_img = ImageTk.PhotoImage(image=pil_img, master=root) #creating image while referring back to previously defined properties

hbar = Scrollbar(root, orient=HORIZONTAL)
hbar.pack(side=BOTTOM, fill=X)
vbar = Scrollbar(root, orient=VERTICAL)
vbar.pack(side=RIGHT, fill=Y) #if image is bigger than moniter than you have problem, thus we make it small in line three and give it scroll bars here

canvas = Canvas(root, width=800, height=600, #smaller than defined before to make room for scroll bars
                scrollregion=(0, 0, canvas_image.shape[1], canvas_image.shape[0]),
                xscrollcommand=hbar.set, yscrollcommand=vbar.set) #just setting up the function of scrolling
canvas.pack(side=LEFT, expand=True, fill=BOTH)

hbar.config(command=canvas.xview)
vbar.config(command=canvas.yview)
canvas.create_image(0, 0, anchor="nw", image=tk_img) #actually puting the image on screen

chosen_x = [0] #defining variables for coordinates
chosen_y = [0]

def give_coords(event):
    true_x = int(canvas.canvasx(event.x)) #these adjust for the fact that you have scrolled which would otherwise not be accounted for
    true_y = int(canvas.canvasy(event.y))
    height, width = canvas_image.shape #pulling the dimensions to account for clicking outside of the image
    
    if 0 <= true_x < width and 0 <= true_y < height: #this is only true if your mouse is within the image so there's no errors when you click off
        pixel_value = image_data[true_y, true_x] #takes the intensity of the pixel from before it was normalized
        print(f"FITS Coord: ({true_x}, {true_y}) | Intensity: {pixel_value}       ", end="\r") #shows everything when you click
        chosen_x[0] = true_x #assigns your mouse coordinates to the previously defined array. Will override everytime you click
        chosen_y[0] = true_y

canvas.bind('<1>', give_coords) #binds function to click, which here is <1>

root.protocol("WM_DELETE_WINDOW", lambda: (root.quit(), root.destroy())) #ends cell rutime when you close out of the image
root.mainloop()

In [ ]:
print(chosen_x) #proof of concept
print(chosen_y)

In [ ]:
hdu_list = fits.open('C:/Users/Astroadmin/cube_norm.fits') #placeholder file - it should take from the created cube when the code gets integrated
data = hdu_list[0].data #pulls cube data from HDU

graph = data[:, chosen_x, chosen_y] #takes a slice for all frames/FIT files at the coordinates you clicked

plt.plot(graph) #plot dot plot
plt.ylabel('Normalized Intensity')
plt.xlabel('frame')
plt.show